# Ghost Holography imaging

Ghost holography imaging (GHI) is an unconventional imaging technique in quantum optics. In this method, a Gaussian random
 X-ray source is splitted to generate two wave beams from this source. The unscattered illumination beam or "reference beam", labeled 1 propagates through a homogeneous or scattering medium up to a high-resolution detector that measures the spatially-integrated resolved transmitted intensity. In contrast, the scattered illumination beam or ``signal beam" labeled  2  propagates through a homogeneous or scattering medium and subsequently interacts with an object (or a mask) to be imaged. The total transmitted intensity is measured by a bucket detector. This method is known as ghost holography imaging simply because one of the detectors (the high-resolution detector) does not see the object to be imaged. However, a high-resolution image of the object is obtained by the cross-correlating the two measured intensity signals.

In ideal noise free regime, the data for each detector is given by  intensity distributions:
\begin{equation}
    I_{\dagger}=|\mathcal{D} (e^{f} u)|^{2},
\end{equation}
where $|\cdot|$ is understood element-wise.  We denote the intensity distribution measured at the bucket detector by
\begin{equation}
    I_{\dagger}^{b}:=\langle 1,I_{\dagger}\rangle.
\end{equation}
 Here $1$ is the constant $1$ function, i.e., 
 $\langle 1,g\rangle=\int_{\mathbb{D}}g(x)dx$.
\begin{equation}
    I_{\dagger}^{\text{ref}}=|(\tilde{\mathcal{D}}u)|^{2}
\end{equation}
represent the intensity distribution of the reference signal at the high-resolution detector.

The forward operator for ghost holography imaging is given by:
\begin{align}
     Tg=C^{\text{obs}},~~~~\text{with}~~Tg=\operatorname{Diag} \left[\tilde{\mathcal{D}} \operatorname{Cov}[u] M_{g} \operatorname{Cov}[u]\tilde{\mathcal{D}}^{*}\right]
 \end{align}
 where $C^{\text{obs}}:=\operatorname{Cov}(I_{\dagger}^{b},I_{\dagger}^{ref})$.
 

In [ ]:
import os
import sys

#sys.path.append(os.path.join(os.path.dirname(__file__), '../'))
#sys.path.append('/home/milad-karimi/itreginstall/itreg/')

from regpy.vecsps import UniformGridFcts
from regpy.solvers.nonlinear.fista import FISTA
from regpy.functionals import QuadraticLowerBound
from regpy.hilbert import L2
from regpy.solvers import Setting
import regpy.stoprules as rules
import matplotlib.pyplot as plt

from auxiliary_ops import  fresnel_prop, Reshape, ReIm
from regpy.operators import RealPart, MatrixMultiplication, SquaredModulus, PtwMultiplication

from create_Vcov import _create_Vcov

import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

# Setting up parameters

In [ ]:
N=256                   #N^2 is the pixel number
M=1                     # number of shots per frame
T=int(1e9)             # the observation time or the number of photon counts
N_b=int(50)             # denotes the rank of the matrix V
sigma=10                # parameter used in the rapid deacaying function to generate modes
N_frame=int(1e3)        # number of frames
fresnel_number=100      # Fresnel number in direction 2 (not properly scaled)
fresnel_number_tilde=100 # Fresnel number in direction 1( not properly scaled)
test_image=1             # choose test images is either 1 or 2

# Discretization of the spaces and Fresnel propagator

In [ ]:
xsample=np.arange(-1,1-1/N,2/N)
ysample=xsample
grid=UniformGridFcts((-1, 1, N), (-1, 1, N), dtype=complex)
grid_2=UniformGridFcts((0, N**2-1, N**2), dtype=complex)
grid_3=UniformGridFcts((0, N_b**2-1, N_b**2), dtype=complex)
codomain=UniformGridFcts((-1, 1, N), (-1, 1, N))
grid_domain=UniformGridFcts((-1, 1, 2), (-1, 1, N), (-1, 1, N))

fp_tilde=fresnel_prop(grid, number=complex(0, 1)/(2*fresnel_number_tilde)) # Fresnel propagator in direction 1
fp=fresnel_prop(grid, number=complex(0, 1)/(2*fresnel_number))          # Fresnel propagator in direction 2

# Compte the factorization  $\operatorname{Cov}[u]=V_{\text{cov}}V_{\text{cov}}^{*}$

In [ ]:
create_types=['spatial', 'Fresnelprop', 'fourier_random']
create_type='fourier_random'

Vcov,S=_create_Vcov(N, N_b, create_type=create_type, grid=grid, sigma=sigma, xsample=xsample, ysample=ysample)
sing=S

We can now introduce the
discrete forward operator
\begin{equation}
    Tg:=\operatorname{Diag}\big[\tilde{\mathcal{D}}\operatorname{Cov}[u]\operatorname{diag}(g)\operatorname{Cov}[u]\tilde{\mathcal{D}}^*\big] \\
    =|\tilde{\mathcal{D}}\operatorname{Cov}[u]|^2 g \approx \operatorname{Corr}.
\end{equation}
Where 
\begin{equation}
\operatorname{Corr}:=\frac{1}{N_{\text{frame}}}\sum_{n=1}^{N_{\text{frame}}}(I_{n}^{\text{b}}-I_{\text{av}}^{\text{b}})(I_{n}^{\text{ref}}-I_{\text{av}}^{\text{ref}})^{\top},~~\text{with}~~I_{\text{av}}^{\text{sig}}:=\frac{1}{N_{\text{frame}}}\sum_{n=1}^{N_{\text{frame}}}I_{n}^{\text{sig}},
\end{equation}
where $\text{sig}\in\{\text{b},\text{ref}\}$.

A key ingredient is the following identity  
\begin{equation}
    |BC^{*}|^{2}=\tau(B)\tau(C)^{*}, 
\end{equation}
where $B,C\in\mathbb{C}^{N_{\text{pix}}\times r}$  and $\tau:\mathbb{C}^{N_{\text{pix}}\times r}\to\mathbb{C}^{N_{\text{pix}}\times r^2}$ is given by $B\mapsto[\tau(B)]_{i,(p,q)}:= B_{ip}\overline{B_{iq}}$.
Therefore, we obtain
\begin{align*}
    Tg=|\tilde{V}V^{*}|^2 g
    =\tau(\tilde{V})\tau(V)^{*}g,~~~\text{with}~~\tilde{V}:=\tilde{\mathcal{D}}V
\end{align*}
 This allows us to compute $T^{*}T$ using matrices of size at 
 most $N_{\text{pix}}\times r^2$.  

In [ ]:
kernel_1=np.zeros((N, N, N_b), dtype=complex)
for i in range(0, N_b):
    kernel_1[:, :, i]=fp_tilde(Vcov[:, :,i])
    
kernel=kernel_1.reshape(N, N, N_b, 1)*kernel_1.conj().reshape(N, N, 1, N_b)
kernel=kernel.reshape(N**2, N_b**2)

mat=(Vcov.T.conj().flatten()).reshape(N_b, 1, N**2)*Vcov.T.reshape(1, N_b, N**2)
mat=mat.reshape(N_b**2, N**2)

reim_1=RealPart(grid).adjoint
resh_1=Reshape(reim_1.codomain, grid_2)
mat_1=MatrixMultiplication(mat, domain=resh_1.codomain, codomain=grid_3)
mat_2=MatrixMultiplication(kernel, domain=mat_1.codomain, codomain=grid_2)
resh_2=Reshape(mat_2.codomain, grid)
reim_2=RealPart(domain=resh_2.codomain)
op=reim_2*resh_2*mat_2*mat_1*resh_1*reim_1

# Create test image

In [ ]:
if test_image == 1:
    X,Y = np.meshgrid(xsample, ysample, sparse=False)
    absorp_0=(abs(X)<0.601)*(abs(Y)<0.199)+(abs(X)<0.199)*(abs(Y)<0.601)
    absorp_0=absorp_0.astype('int')
    absorp_1=(X**2+Y**2<=0.501**2)*(X**2+Y**2>=0.45**2)
    absorp_1=absorp_1.astype('int')
    absorp_2=(abs(X)<0.3)*(abs(Y)<0.055)+(abs(X)<0.055)*(abs(Y)<0.3)
    absorp_2=absorp_2.astype('int')
    absorp_3=(abs(X)**25+abs(Y)**25<=0.601**25)*(abs(X)**25+abs(Y)**25>=0.551**25)
    absorp_3=absorp_3.astype('int')
    absorp=absorp_0+absorp_1+absorp_2+absorp_3
    support_mask=((abs(X)<=0.601)*(abs(Y)<=0.601)).astype('int')
    contrast = support_mask*(0.1*absorp )
    y=op(np.exp(2*contrast)) 
    mask=(contrast!=0)                                         # defines the support of the contrast
elif test_image == 2:
    X,Y = np.meshgrid(xsample, ysample, sparse=False)
    absorp=np.load('cell256.npy')
    #support_mask=((abs(X)<=0.801)*(abs(Y)<=0.801)).astype('int')              #constant rectangular bump
    support_mask=((abs(X)**2+abs(Y)**2)<=0.6).astype('int')                    # constant circular bump
    contrast = 0.02*support_mask+(0.1*absorp)
    y=op(np.exp(2*contrast)) 
    mask=(contrast!=0) 
else:
    raise ValueError

# Create shot noise

In [ ]:
T=1e3
ptw_detection= SquaredModulus(grid)
mult=PtwMultiplication(grid, np.exp(contrast))
ptw_op=ptw_detection*fp*mult

ptw_detection= SquaredModulus(grid)
ptw_op_tilde=ptw_detection*fp_tilde

uincmat=1/np.sqrt(2)*np.random.randn(N_frame*M, N_b)+complex(0,1)*1/np.sqrt(2)*np.random.randn(N_frame*M, N_b)
uincmat=uincmat.reshape(N_frame, M, N_b)

corr_signal=np.zeros((N**2))
intens_tot=0
intens_tot_tilde=np.zeros(N**2)
for i in range(0, N_frame):
    print(i)
    signal=0
    signal_tilde=np.zeros((N**2))
    for j in range(0, M):
        uinc=Vcov.dot(uincmat[i,j, :]).reshape(N, N)
        #Shot noise
        signal+=np.sum(ptw_op(uinc))
        signal_tilde+=ptw_op_tilde(uinc).reshape(N**2)
    
    #Cox-processes
    signal=(1/T)*np.random.poisson(lam=T*signal.flatten(), size=(N**2))
    signal_tilde=(1/T)*np.random.poisson(lam=T*signal_tilde.flatten(), size=(N**2))
    intens_tot+=signal
    intens_tot_tilde+=signal_tilde
    corr_signal+=signal*signal_tilde.conj()

# Create data

In [ ]:
crosscor=(corr_signal-intens_tot*intens_tot_tilde/N_frame)/N_frame

crosscor=crosscor.reshape(N, N)
exact_data=op(np.exp(2*contrast))
noise=crosscor-exact_data
noiselevel=np.sqrt(noise)
fac=0.0001       #Approximates N_frame*1/fac**2 frames
data=exact_data + fac*noise

# Inversion method with FISTA algorithm

In [ ]:
Nfista=100               # number of FISTA iterations 
regpar_GHI=1e-10         # regularization parameter

ReIm_op=ReIm(grid)
data_space = L2(op.codomain)

#penalty=QuadraticNonneg(op.domain)    # non-negativity constriant
penalty=QuadraticLowerBound(op.domain,lb=1, x0=op.domain.ones())    # non-negativity constriant
penalty1=L2(op.domain)
setting = Setting(op=op, penalty=penalty, data_fid = data_space, data=data,regpar=regpar_GHI)
FISTA_solver = FISTA(setting)
stoprule = (rules.CountIterations(Nfista))

reco, reco_data=FISTA_solver.run(stoprule)
reco=1/2*np.log(reco)

# Plots

In [ ]:
plt.rcParams['font.family'] = 'Serif'
fontsize=16

plt.figure()
plt.imshow(uinc.real)
plt.colorbar()
plt.title('Sample Incident Field',fontsize=fontsize)
plt.show()


plt.figure()
plt.imshow(intens_tot_tilde.reshape(N,N))
plt.colorbar()
plt.title('Reference intensity',fontsize=fontsize)
plt.show()


plt.figure()
plt.imshow(intens_tot.reshape(N,N))
plt.colorbar()
plt.title('Bucket intensity',fontsize=fontsize)
plt.show()


plt.figure()
plt.imshow(reco.real, vmin=0,vmax=0.2)
plt.colorbar()
plt.title('Recovered absorption',fontsize=fontsize)
plt.show()

plt.figure()
plt.imshow(contrast)
plt.colorbar()
plt.title('Exact absorption',fontsize=fontsize)
plt.show()

plt.figure()
plt.plot(reco[150, :], label='Recovered absorption')
plt.plot(contrast[150, :], label='Exact absorption')
plt.legend()
plt.show()